# Training an Encrypted Neural Network

In this tutorial, we will walk through an example of how we can train a neural network with CrypTen. This is particularly relevant for the <i>Feature Aggregation</i>, <i>Data Labeling</i> and <i>Data Augmentation</i> use cases. We will focus on the usual two-party setting and show how we can train an accurate neural network for digit classification on the MNIST data.

For concreteness, this tutorial will step through the <i>Feature Aggregation</i> use cases: Alice and Bob each have part of the features of the data set, and wish to train a neural network on their combined data, while keeping their data private. 

## Setup
As usual, we'll begin by importing and initializing the `crypten` and `torch` libraries.  

We will use the MNIST dataset to demonstrate how Alice and Bob can learn without revealing protected information. For reference, the feature size of each example in the MNIST data is `28 x 28`. Let's assume Alice has the first `28 x 20` features and Bob has last `28 x 8` features. One way to think of this split is that Alice has the (roughly) top 2/3rds of each image, while Bob has the bottom 1/3rd of each image. We'll again use our helper script `mnist_utils.py` that downloads the publicly available MNIST data, and splits the data as required.

For simplicity, we will restrict our problem to binary classification: we'll simply learn how to distinguish between 0 and non-zero digits. For speed of execution in the notebook, we will only create a dataset of a 100 examples.

In [1]:
import crypten
import torch
import time
import multiprocessing as mp

mp.set_start_method('spawn')
crypten.init()
torch.set_num_threads(1)

[W ProcessGroupGloo.cpp:723] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


In [2]:
%run ./mnist_utils.py --option features --reduced 100 --binary

/opt/homebrew/Caskroom/miniconda/base/envs/lora_mps/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Next, we'll define the network architecture below, and then describe how to train it on encrypted data in the next section. 

In [3]:
import torch.nn as nn
import torch.nn.functional as F

#Define an example network
class ExampleNet(nn.Module):
    def __init__(self):
        super(ExampleNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=0)
        self.fc1 = nn.Linear(16 * 12 * 12, 100)
        self.fc2 = nn.Linear(100, 2) # For binary classification, final layer needs only 2 outputs
 
    def forward(self, x):
        out = self.conv1(x)
        out = F.relu(out)
        out = F.max_pool2d(out, 2)
        out = out.view(-1, 16 * 12 * 12)
        out = self.fc1(out)
        out = F.relu(out)
        out = self.fc2(out)
        return out
    
crypten.common.serial.register_safe_class(ExampleNet)

## Encrypted Training

After all the material we've covered in earlier tutorials, we only need to know a few additional items for encrypted training. We'll first discuss how the training loop in CrypTen differs from PyTorch. Then, we'll go through a complete example to illustrate training on encrypted data from end-to-end.

### How does CrypTen training differ from PyTorch training?

There are two main ways implementing a CrypTen training loop differs from a PyTorch training loop. We'll describe these items first, and then illustrate them with small examples below.

<i>(1) Use one-hot encoding</i>: CrypTen training requires all labels to use one-hot encoding. This means that when using standard datasets such as MNIST, we need to modify the labels to use one-hot encoding.

<i>(2) Directly update parameters</i>: CrypTen does not use the PyTorch optimizers. Instead, CrypTen implements encrypted SGD by implementing its own `backward` function, followed by directly updating the parameters. As we will see below, using SGD in CrypTen is very similar to using the PyTorch optimizers.

We now show some small examples to illustrate these differences. As before, we will assume Alice has the rank 0 process and Bob has the rank 1 process.

In [4]:
# Define source argument values for Alice and Bob
ALICE = 0
BOB = 1

In [5]:
# Load Alice's data 
data_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)

In [6]:
# We'll now set up the data for our small example below
# For illustration purposes, we will create toy data
# and encrypt all of it from source ALICE
x_small = torch.rand(100, 1, 28, 28)
y_small = torch.randint(1, (100,))

# Transform labels into one-hot encoding
label_eye = torch.eye(2)
y_one_hot = label_eye[y_small]

# Transform all data to CrypTensors
x_train = crypten.cryptensor(x_small, src=ALICE)
y_train = crypten.cryptensor(y_one_hot)

# Instantiate and encrypt a CrypTen model
model_plaintext = ExampleNet()
dummy_input = torch.empty(1, 1, 28, 28)
a_model = crypten.nn.from_pytorch(model_plaintext, dummy_input)

aa =1

========== Diagnostic Run torch.onnx.export version 2.1.0.dev20230706 ==========
verbose: False, log level: 40
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

========== Diagnostic Run torch.onnx.export version 2.1.0.dev20230706 ==========
verbose: False, log level: 40
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



/opt/homebrew/Caskroom/miniconda/base/envs/lora_mps/lib/python3.8/site-packages/crypten-0.4.0-py3.8.egg/crypten/nn/onnx_converter.py:176: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:212.)
  param = torch.from_numpy(numpy_helper.to_array(node))


In [7]:
a_model.encrypt()

# Train the model
a_model.train()

# Set up the optimizer
optimizer = crypten.optim.SGD(a_model.parameters(), lr=0.1)
loss = crypten.nn.MSELoss() # Choose loss functions

# Train the model
start = time.time()
for i in range(10):
    optimizer.zero_grad()
    output = a_model(x_train)
    loss_value = loss(output, y_train)
    loss_value.backward()
    param_after_backwrd = [ param.grad.clone() for param in a_model.parameters()]
    optimizer.step()
    print("Epoch", i, "Loss", loss_value.get_plain_text())
end = time.time()
print("Training time:", end - start, "seconds")


Epoch 0 Loss tensor(0.4488)
Epoch 1 Loss tensor(0.1393)
Epoch 2 Loss tensor(0.3370)
Epoch 3 Loss tensor(0.1326)
Epoch 4 Loss tensor(0.0043)
Epoch 5 Loss tensor(0.0005)
Epoch 6 Loss tensor(0.0003)
Epoch 7 Loss tensor(0.0003)
Epoch 8 Loss tensor(0.0003)
Epoch 9 Loss tensor(0.0003)
Training time: 10.46775221824646 seconds


In [42]:
import crypten.nn as cnn

class ExampleNetCrypt(cnn.Module):
    def __init__(self):
        super(ExampleNetCrypt, self).__init__()
        self.conv1 = cnn.Conv2d(1, 16, kernel_size=5, padding=0)
        self.fc1 = cnn.Linear(16 * 12 * 12, 100, bias=False)

        #self.fc1 = cnn.Embedding(16 * 12 * 12, 100)
        self.layernorm = cnn.LayerNorm(100)
        #self.batchnorm = cnn.BatchNorm1d(100)
        
        #self.fc2 = cnn.Linear(100, 2) # For binary classification, final layer needs only 2 outputs
        
        self.fc2 = cnn.Embedding(100, 2)
        self.fc3 = cnn.Embedding(2, 2)
        self.fc4 = cnn.Embedding(2, 2)
        self.layernorm2 = cnn.LayerNorm(2)
        self.relu = cnn.ReLU()
        self.max_pool2d = cnn.MaxPool2d(2)
        self.dropout = cnn.Dropout(0.5)

 
    def forward(self, x):
        out = self.conv1(x)
        out = self.relu(out)
        out = self.max_pool2d(out)
        out = out.view(-1, 16 * 12 * 12)
        out = self.fc1(out)
        #out = self.batchnorm(out)
        out = self.layernorm(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.fc3(out)
        out = self.fc4(out)
        out = self.layernorm2(out)
        out = self.dropout(out)
        return out
    

class CryptenClassifier(cnn.Module):
    def __init__(self):
        super(CryptenClassifier, self).__init__()
        self.num_labels = 2
        self.pre_classifier = cnn.Linear(768, 768)
        self.classifier = cnn.Linear(768, 2)
        self.dropout = cnn.Dropout(0.1)
        self.classifier_act_fn = cnn.ReLU()
        self.loss = cnn.CrossEntropyLoss()
    
    def forward(self, input, labels):

        hidden_state = input[0]  # (bs, seq_len, dim)
        #print(hidden_state.shape)
        pooled_output = hidden_state[:, 0, :]  # (bs, dim)
        #print(pooled_output.shape)
        pooled_output = self.pre_classifier(pooled_output)  # (bs, dim)
        pooled_output = self.classifier_act_fn(pooled_output)  # (bs, dim)
        pooled_output = self.dropout(pooled_output)  # (bs, dim)
        logits = self.classifier(pooled_output)  # (bs, num_labels)
        logits = logits.view(-1, self.num_labels)
        loss = self.loss(logits, labels)
        return loss, logits

In [10]:
#!/usr/bin/env python3

# Copyright (c) Facebook, Inc. and its affiliates.
#
# This source code is licensed under the MIT license found in the
# LICENSE file in the root directory of this source tree.

import crypten

from crypten.optim import Optimizer


class SGD(Optimizer):
    r"""Implements stochastic gradient descent (optionally with momentum).
    Nesterov momentum is based on the formula from
    `On the importance of initialization and momentum in deep learning`__.
    Args:
        params (iterable): iterable of parameters to optimize or dicts defining
            parameter groups
        lr (float): learning rate
        momentum (float, optional): momentum factor (default: 0)
        weight_decay (float, optional): weight decay (L2 penalty) (default: 0)
        dampening (float, optional): dampening for momentum (default: 0)
        nesterov (bool, optional): enables Nesterov momentum (default: False)
        grad_threshold (float, optional): imposes a threshold on the magnitude of gradient values.
            Gradient values with magnitude above the threshold will be replaced with 0.
    Example:
        >>> optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
        >>> optimizer.zero_grad()
        >>> loss_fn(model(input), target).backward()
        >>> optimizer.step()
    __ http://www.cs.toronto.edu/%7Ehinton/absps/momentum.pdf
    .. note::
        The implementation of SGD with Momentum/Nesterov subtly differs from
        Sutskever et. al. and implementations in some other frameworks.
        Considering the specific case of Momentum, the update can be written as
        .. math::
            \begin{aligned}
                v_{t+1} & = \mu * v_{t} + g_{t+1}, \\
                p_{t+1} & = p_{t} - \text{lr} * v_{t+1},
            \end{aligned}
        where :math:`p`, :math:`g`, :math:`v` and :math:`\mu` denote the
        parameters, gradient, velocity, and momentum respectively.
        This is in contrast to Sutskever et. al. and
        other frameworks which employ an update of the form
        .. math::
            \begin{aligned}
                v_{t+1} & = \mu * v_{t} + \text{lr} * g_{t+1}, \\
                p_{t+1} & = p_{t} - v_{t+1}.
            \end{aligned}
        The Nesterov version is analogously modified.
    """

    def __init__(
        self,
        params,
        lr,
        momentum=0,
        dampening=0,
        weight_decay=0,
        nesterov=False,
        grad_threshold=None,
    ):
        if not isinstance(lr, (int, float)) or lr < 0.0:
            raise ValueError("Invalid learning rate: {}".format(lr))
        if not isinstance(momentum, (int, float)) or momentum < 0.0:
            raise ValueError("Invalid momentum value: {}".format(momentum))
        if not isinstance(dampening, (int, float)):
            raise ValueError("Invalid dampening value {}".format(dampening))
        if not isinstance(weight_decay, (int, float)) or weight_decay < 0.0:
            raise ValueError("Invalid weight_decay value: {}".format(weight_decay))

        defaults = {
            "lr": lr,
            "momentum": momentum,
            "dampening": dampening,
            "weight_decay": weight_decay,
            "nesterov": nesterov,
        }
        if nesterov and (momentum <= 0 or dampening != 0):
            raise ValueError("Nesterov momentum requires a momentum and zero dampening")

        # Compute thresholding based on square value since abs is more expensive
        self.square_threshold = grad_threshold
        if self.square_threshold is not None:
            self.square_threshold *= self.square_threshold

        super(SGD, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(SGD, self).__setstate__(state)
        for group in self.param_groups:
            group.setdefault("nesterov", False)

    def step(self, closure=None):
        """Performs a single optimization step.
        Arguments:
            closure (callable, optional): A closure that reevaluates the model
                and returns the loss.
        """
        with crypten.no_grad():
            loss = None
            if closure is not None:
                with crypten.enable_grad():
                    loss = closure()

            for group in self.param_groups:
                weight_decay = group["weight_decay"]
                momentum = group["momentum"]
                dampening = group["dampening"]
                nesterov = group["nesterov"]

                for p in group["params"]:
                    if p.grad is None:
                        continue

                    # Threshold gradients to prevent gradient explosion
                    if self.square_threshold is not None:
                        d_p = p.grad.mul(p.grad.square().lt(self.square_threshold))
                    else:
                        d_p = p.grad

                    if weight_decay != 0:
                        d_p = d_p.add(p.mul(weight_decay))
                    if momentum != 0:
                        param_state = self.state[id(p)]
                        if "momentum_buffer" not in param_state:
                            buf = param_state["momentum_buffer"] = d_p.clone().detach()
                        else:
                            buf = param_state["momentum_buffer"]
                            buf.mul_(momentum).add_(d_p.mul(1 - dampening))
                        if nesterov:
                            d_p = d_p.add(buf.mul(momentum))
                        else:
                            d_p = buf

                    p.sub_(d_p.mul(group["lr"]))

            return loss


In [43]:
classifier = CryptenClassifier()

classifier.train()
classifier.encrypt()

loss = cnn.CrossEntropyLoss()
optimizer = SGD(classifier.parameters(), lr=0.05)

x_class =  torch.rand(1, 1, 128, 768)
x_class = crypten.cryptensor(x_class, src=ALICE)
y_class = torch.tensor([1])
y_one_hot = label_eye[y_class]
y_class = crypten.cryptensor(y_one_hot)

# Train the model
for i in range(10):
    optimizer.zero_grad()
    #output = classifier(x_class)
    #loss_value = output[0]
    #logits = output[1]
    loss_value, logits = classifier(x_class, y_class)
    #output = output.view(-1, 2)
    #loss_value = loss(output.view(-1, 2), y_class)
    loss_value.backward()
    optimizer.step()
    print("Epoch", i, "Loss", loss_value.get_plain_text())

Epoch 0 Loss tensor(0.3733)
Epoch 1 Loss tensor(0.0760)
Epoch 2 Loss tensor(0.0549)
Epoch 3 Loss tensor(0.0575)
Epoch 4 Loss tensor(0.0444)
Epoch 5 Loss tensor(0.0383)
Epoch 6 Loss tensor(0.0380)
Epoch 7 Loss tensor(0.0344)
Epoch 8 Loss tensor(0.0353)
Epoch 9 Loss tensor(0.0315)


In [44]:
# Example: Stochastic Gradient Descent in CrypTen
a_model.encrypt()
model = ExampleNetCrypt() # Instantiate model


model.train() # Change to training mode
model.encrypt() # Encrypt model


#loss = crypten.nn.MSELoss() # Choose loss functions

loss = crypten.nn.MSELoss() # Choose loss functions
optimizer = SGD(model.parameters(), lr=5e-5) # Choose optimizer


# Set parameters: learning rate, num_epochs
learning_rate = 0.001
num_epochs = 6

tic = time.time()
# Train the model: SGD on encrypted data
for i in range(num_epochs):

    # forward pass
    output = model(x_train)
    loss_value = loss(output, y_train)
    
    # set gradients to zero
    model.zero_grad()

    # perform backward pass
    param_before_backward = [param.clone() for param in model.parameters()]
    loss_value.backward()
    param_after_backward = [param.clone() for param in model.parameters()]
    #grad = loss_value.grad
    #print(grad.get_plain_text())
    #param_grads = [param.grad for param in model.parameters()]

    # update parameters
    optimizer.step()
    #model.update_parameters(learning_rate) 
    
    # examine the loss after each epoch
    print("Epoch: {0:d} Loss: {1:.4f}".format(i, loss_value.get_plain_text()))
    
toc = time.time()
print(toc-tic)

Epoch: 0 Loss: 2.4041
Epoch: 1 Loss: 2.4034
Epoch: 2 Loss: 2.3907
Epoch: 3 Loss: 2.2565
Epoch: 4 Loss: 2.1916
Epoch: 5 Loss: 1.9173
35.6727249622345


In [8]:
# Example: Stochastic Gradient Descent in CrypTen
model.cuda("cuda:2")
model.train() # Change to training mode
loss = crypten.nn.MSELoss() # Choose loss functions

# Set parameters: learning rate, num_epochs
learning_rate = 0.001
num_epochs = 2
x_train.cuda("cuda:2")
y_train.cuda("cuda:2")

tic = time.time()
# Train the model: SGD on encrypted data
for i in range(num_epochs):

    # forward pass
    output = model(x_train)
    loss_value = loss(output, y_train)
    
    # set gradients to zero
    model.zero_grad()

    # perform backward pass
    loss_value.backward()

    # update parameters
    model.update_parameters(learning_rate) 
    
    # examine the loss after each epoch
    print("Epoch: {0:d} Loss: {1:.4f}".format(i, loss_value.get_plain_text()))
    toc = time.time()
print(toc-tic)

Epoch: 0 Loss: 0.3942
Epoch: 1 Loss: 0.3712
2.94539213180542


### A Complete Example

We now put these pieces together for a complete example of training a network in a multi-party setting. 

As in Tutorial 3, we'll assume Alice has the rank 0 process, and Bob has the rank 1 process; so we'll load and encrypt Alice's data with `src=0`, and load and encrypt Bob's data with `src=1`. We'll then initialize a plaintext model and convert it to an encrypted model, just as we did in Tutorial 4. We'll finally define our loss function, training parameters, and run SGD on the encrypted data. For the purposes of this tutorial we train on 100 samples; training should complete in ~3 minutes per epoch.

In [2]:
import crypten
import torch
import crypten.communicator as comm
import crypten.mpc as mpc

import multiprocessing as mp

mp.set_start_method('fork')
crypten.init()



[W ProcessGroupGloo.cpp:723] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


In [9]:
@mpc.run_multiprocess(world_size=2)
def run_random_subsampling():
    len = 1000000
    rank = crypten.communicator.get().get_rank()

    x = torch.randn(len, 1)
    x_enc = crypten.cryptensor(x, src=0)

    sedd = rank + 1
    #torch.manual_seed(sedd)
    #idx = torch.randperm(len)
    #crypten.print(f"Party {rank} has idx_clear: {idx}", in_order=True)
    #idx = torch.randperm(len)
    #crypten.print(f"Party {rank} has idx_clear: {idx}", in_order=True)
    #idx = torch.randperm(len)
    #crypten.print(f"Party {rank} has idx_clear: {idx}", in_order=True)

    #idx_enc = crypten.cryptensor(idx, src=0)

    #idx_clear = idx_enc.get_plain_text()
    #crypten.print(f"Party {rank} has idx_clear: {idx_clear}", in_order=True)

    sample_rate = 256 / len
    sample_rates = torch.tensor([sample_rate] * len)
    print(sample_rates.shape)
    poisson_samples = torch.poisson(sample_rates)
    poisson_samples_enc = crypten.cryptensor(poisson_samples)
    poisson_samples_clear = poisson_samples_enc.get_plain_text()
    mask = poisson_samples_clear > 0
    # count the true masks
    count = mask.sum()
    crypten.print(f"Party {rank} has count: {count}", in_order=True)
    crypten.print(f"Party {rank} has mask: {x_enc[mask]}", in_order=True)


run_random_subsampling()

torch.Size([1000000])torch.Size([1000000])

Party 0 has count: 271
Party 1 has count: 271
Party 0 has mask: MPCTensor(
	_tensor=tensor([[-5073942953948907417],
        [ 4654598180266939191],
        [-3541063178636747720],
        [ 4906223845085443107],
        [ 7541286581795982125],
        [ 7594640731127018474],
        [-4531614700917024333],
        [ 8916724114805997295],
        [  306002549776704070],
        [-3414513402579831180],
        [ 8867965579114011501],
        [ 1120670850856651413],
        [ 4145443687395581734],
        [-5601359859239785673],
        [-4316929175816680339],
        [-1254150481061805063],
        [ 7677276302501325819],
        [ 4652237997598065085],
        [ 1133042313252174554],
        [ 8003695478865438943],
        [ 3662219001882293848],
        [ 5759637677485224474],
        [-2847152635409978286],
        [ 5039383121041684607],
        [-7964817412365110201],
        [ 6232773586393707507],
        [ 8306251417479513395],
        

[None, None]

In [9]:
import crypten.mpc as mpc
import crypten.communicator as comm

# Convert labels to one-hot encoding
# Since labels are public in this use case, we will simply use them from loaded torch tensors
labels = torch.load('/tmp/train_labels.pth')
labels = labels.long()
labels_one_hot = label_eye[labels]

@mpc.run_multiprocess(world_size=2)
def run_cuda_encrypted_training():
    # Load data:
    x_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE).cuda("cuda:2")
    x_bob_enc = crypten.load_from_party('/tmp/bob_train.pth', src=BOB).cuda("cuda:2")
    
    crypten.print(x_alice_enc.size())
    crypten.print(x_bob_enc.size())
    
    # Combine the feature sets: identical to Tutorial 3
    x_combined_enc = crypten.cat([x_alice_enc, x_bob_enc], dim=2)
    
    # Reshape to match the network architecture
    x_combined_enc = x_combined_enc.unsqueeze(1)
    
    
    # Commenting out due to intermittent failure in PyTorch codebase
    
    # Initialize a plaintext model and convert to CrypTen model
    pytorch_model = ExampleNet()
    model = crypten.nn.from_pytorch(pytorch_model, dummy_input)
    model.encrypt()
    model.cuda("cuda:2")
    # Set train mode
    model.train()
  
    # Define a loss function
    loss = crypten.nn.MSELoss()

    # Define training parameters
    learning_rate = 0.001
    num_epochs = 2
    batch_size = 10
    num_batches = x_combined_enc.size(0) // batch_size
    
    rank = comm.get().get_rank()
    for i in range(num_epochs): 
        crypten.print(f"Epoch {i} in progress:")       
        
        for batch in range(num_batches):
            # define the start and end of the training mini-batch
            start, end = batch * batch_size, (batch + 1) * batch_size
                                    
            # construct CrypTensors out of training examples / labels
            x_train = x_combined_enc[start:end]
            y_batch = labels_one_hot[start:end]
            y_train = crypten.cryptensor(y_batch, requires_grad=True)
            
            # perform forward pass:
            output = model(x_train)
            loss_value = loss(output, y_train)
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value forward: {loss_value}", in_order=True)
            # set gradients to "zero" 
            model.zero_grad()

            # perform backward pass: 
            loss_value.backward()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value backward: {model.}", in_order=True)


            # update parameters
            model.update_parameters(learning_rate)
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tModel: {model}", in_order=True)

            
            # Print progress every batch:
            batch_loss = loss_value.get_plain_text()
            crypten.print(f"\tBatch {(batch + 1)} of {num_batches} Loss {batch_loss.item():.4f}", in_order=True)
        #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tLoss value: {batch_loss}", in_order=True)

tic = time.time()
run_cuda_encrypted_training()
toc = time.time()
print(f"Cuda time:{toc-tic}")

PicklingError: Can't pickle <function run_encrypted_training at 0x7ff87068a550>: it's not the same object as __main__.run_encrypted_training

In [ ]:
import crypten.mpc as mpc
import crypten.communicator as comm

# Convert labels to one-hot encoding
# Since labels are public in this use case, we will simply use them from loaded torch tensors
labels = torch.load('/tmp/train_labels.pth')
labels = labels.long()
labels_one_hot = label_eye[labels]

@mpc.run_multiprocess(world_size=2)
def run_encrypted_training():
    # Load data:
    x_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)
    x_bob_enc = crypten.load_from_party('/tmp/bob_train.pth', src=BOB)
    
    crypten.print(x_alice_enc.size())
    crypten.print(x_bob_enc.size())
    
    # Combine the feature sets: identical to Tutorial 3
    x_combined_enc = crypten.cat([x_alice_enc, x_bob_enc], dim=2)
    
    # Reshape to match the network architecture
    x_combined_enc = x_combined_enc.unsqueeze(1)
    
    
    # Commenting out due to intermittent failure in PyTorch codebase
    
    # Initialize a plaintext model and convert to CrypTen model
    pytorch_model = ExampleNet()
    model = crypten.nn.from_pytorch(pytorch_model, dummy_input)
    model.encrypt()
    # Set train mode
    model.train()
  
    # Define a loss function
    loss = crypten.nn.MSELoss()

    # Define training parameters
    learning_rate = 0.001
    num_epochs = 2
    batch_size = 10
    num_batches = x_combined_enc.size(0) // batch_size
    
    rank = comm.get().get_rank()
    for i in range(num_epochs): 
        crypten.print(f"Epoch {i} in progress:")       
        
        for batch in range(num_batches):
            # define the start and end of the training mini-batch
            start, end = batch * batch_size, (batch + 1) * batch_size
                                    
            # construct CrypTensors out of training examples / labels
            x_train = x_combined_enc[start:end]
            y_batch = labels_one_hot[start:end]
            y_train = crypten.cryptensor(y_batch, requires_grad=True)
            
            # perform forward pass:
            output = model(x_train)
            loss_value = loss(output, y_train)
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value forward: {loss_value}", in_order=True)
            # set gradients to "zero" 
            model.zero_grad()

            # perform backward pass: 
            loss_value.backward()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value backward: {model.}", in_order=True)


            # update parameters
            model.update_parameters(learning_rate)
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tModel: {model}", in_order=True)

            
            # Print progress every batch:
            batch_loss = loss_value.get_plain_text()
            crypten.print(f"\tBatch {(batch + 1)} of {num_batches} Loss {batch_loss.item():.4f}", in_order=True)
        #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tLoss value: {batch_loss}", in_order=True)

tic = time.time()
run_encrypted_training()
toc = time.time()
print(f"CPU time:{toc-tic}")

We see that the average batch loss decreases across the epochs, as we expect during training.

This completes our tutorial. Before exiting this tutorial, please clean up the files generated using the following code.

In [ ]:
import os

filenames = ['/tmp/alice_train.pth', 
             '/tmp/bob_train.pth', 
             '/tmp/alice_test.pth',
             '/tmp/bob_test.pth', 
             '/tmp/train_labels.pth',
             '/tmp/test_labels.pth']

for fn in filenames:
    if os.path.exists(fn): os.remove(fn)